In [37]:
from collections import defaultdict

import matplotlib.pyplot as plt
import torch
from tensordict.nn import TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor
from torch import nn

from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (
    Compose,
    DoubleToFloat,
    ObservationNorm,
    StepCounter,
    TransformedEnv,
)
from torchrl.envs.libs.gym import GymEnv
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm
import multiprocessing

In [4]:
# Define hyperparameters

is_fork = multiprocessing.get_start_method() == "fork"
device = (
    torch.device(0)
    if torch.cuda.is_available() and not is_fork
    else torch.device("cpu")
)
num_cells = 256  # number of cells in each layer i.e. output dim.
lr = 3e-4
max_grad_norm = 1.0

In [5]:
# Data collection parameters

frames_per_batch = 1000
total_frames = 10_000

In [6]:
# PPO parameters

sub_batch_size = 64  # cardinality of the sub-samples gathered from the current data in the inner loop
num_epochs = 10  # optimization steps per batch of data collected
clip_epsilon = (
    0.2  # clip value for PPO loss: see the equation in the intro for more context.
)
gamma = 0.99
lmbda = 0.95
entropy_eps = 1e-4

In [ ]:
# Env 
# Alternatively, one could also directly create a gym environment 
# using gym.make(env_name, **kwargs) and wrap it in a GymWrapper class.

base_env = GymEnv("Pendulum", device=device) # InvertedDoublePendulum gives error (mujoco)

In [23]:
# To add transforms to an environment one can wrap it in a TransformedEnv instnace
# First to encode is a Normalization transform

env = TransformedEnv(
    base_env,
    Compose(
        ObservationNorm(in_keys=["observation"]),
        DoubleToFloat(),
        StepCounter(),
    ),
)

In [24]:
env.transform[0].init_stats(num_iter=1000, reduce_dim=0, cat_dim=0)


In [25]:
print("normalization constant shape:", env.transform[0].loc.shape)

normalization constant shape: torch.Size([3])


In [26]:
print("observation_spec:", env.observation_spec)
print("reward_spec:", env.reward_spec)
print("input_spec:", env.input_spec)
print("action_spec (as defined by input_spec):", env.action_spec)

observation_spec: Composite(
    observation: BoundedContinuous(
        shape=torch.Size([3]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([3]), device=cpu, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([3]), device=cpu, dtype=torch.float32, contiguous=True)),
        device=cpu,
        dtype=torch.float32,
        domain=continuous),
    step_count: BoundedDiscrete(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, contiguous=True),
            high=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, contiguous=True)),
        device=cpu,
        dtype=torch.int64,
        domain=discrete),
    device=cpu,
    shape=torch.Size([]))
reward_spec: UnboundedContinuous(
    shape=torch.Size([1]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, contiguous=True),
        high=Tensor(sha

In [ ]:
# the check_env_specs() function runs a small rollout and compares its output 
# against the environment specs. If no error is raised, 
# we can be confident that the specs are properly defined:

check_env_specs(env)

2025-05-15 15:32:39,965 [torchrl][INFO] check_env_specs succeeded!


In [28]:
# For fun, let’s see what a simple random rollout looks like. You can call env.rollout(n_steps) 
# and get an overview of what the environment inputs and outputs look like. Actions will automatically
#  be drawn from the action spec domain, so you don’t need to care about designing a random sampler.

rollout = env.rollout(3)
print("rollout of three steps:", rollout)
print("Shape of the rollout TensorDict:", rollout.batch_size)

rollout of three steps: TensorDict(
    fields={
        action: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([3, 3]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                step_count: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.int64, is_shared=False),
                terminated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False)},
            batch_size=torch.Size([3]),
            devic

In [29]:
actor_net = nn.Sequential(
    nn.LazyLinear(num_cells, device=device), # Only specify out features
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(2 * env.action_spec.shape[-1], device=device),
    NormalParamExtractor(),
)

# NormalParamExtractor() is a utility module in TorchRL (PyTorch Reinforcement Learning library), 
# not native PyTorch. It’s typically used to transform the output of a neural network into the 
# parameters of a Normal distribution — specifically, the mean (loc) and standard deviation (scale).

In [30]:
# To enable the policy to “talk” with the environment through the tensordict 
# data carrier, we wrap the nn.Module in a TensorDictModule. This class will 
# simply ready the in_keys it is provided with and write the outputs in-place at 
# the registered out_keys.

policy_module = TensorDictModule(
    actor_net, in_keys = ["observation"], out_keys=["loc", "scale"]
)

In [31]:
env.action_spec

BoundedContinuous(
    shape=torch.Size([1]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, contiguous=True),
        high=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, contiguous=True)),
    device=cpu,
    dtype=torch.float32,
    domain=continuous)

In [32]:
# We now need to build a distribution out of the location and scale of our normal
#  distribution. To do so, we instruct the ProbabilisticActor class to build a TanhNormal 
# out of the location and scale parameters. We also provide the minimum and maximum values 
# of this distribution, which we gather from the environment specs.

#The name of the in_keys (and hence the name of the out_keys from the TensorDictModule above) 
# cannot be set to any value one may like, as the TanhNormal distribution constructor will expect 
# the loc and scale keyword arguments. That being said, ProbabilisticActor also accepts Dict[str, str]
#  typed in_keys where the key-value pair indicates what in_key string should be used for every keyword
#  argument that is to be used.

policy_module = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec,
    in_keys = ["loc", "scale"],
    distribution_class=TanhNormal,
    distribution_kwargs={
        "low": env.action_spec.space.low,
        "high": env.action_spec.space.high,
    },
    return_log_prob = True
    # we'll need the log-prob for the numerator of the importance weights
)

In [33]:
# Value network

value_net = nn.Sequential(
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(1, device=device),
)

value_module = ValueOperator(
    module = value_net,
    in_keys=["observation"],
)

In [34]:
print("Running policy:", policy_module(env.reset()))
print("Running value:", value_module(env.reset()))

Running policy: TensorDict(
    fields={
        action: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        loc: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, is_shared=False),
        observation: Tensor(shape=torch.Size([3]), device=cpu, dtype=torch.float32, is_shared=False),
        sample_log_prob: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.float32, is_shared=False),
        scale: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, is_shared=False),
        step_count: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, is_shared=False),
        terminated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        truncated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False)},
    batch_size=torch.Size([]),
    device=cpu,
    is_shared=False)
Running va

In [35]:
# Data collector

# TorchRL provides a set of DataCollector classes. Briefly, these classes execute three operations: 
# reset an environment, compute an action given the latest observation, execute a step in the 
# environment, and repeat the last two steps until the environment signals a stop (or reaches a 
# done state).

# The simplest data collector is the SyncDataCollector: it is an iterator that you can use to get 
# batches of data of a given length, and that will stop once a total number of frames (total_frames) 
# have been collected. Other data collectors (MultiSyncDataCollector and MultiaSyncDataCollector) 
# will execute the same operations in synchronous and asynchronous manner over a set of multiprocessed 
# workers.

collector = SyncDataCollector(
    env,
    policy_module,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
    split_trajs=False,
    device=device,
)

In [38]:
replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=frames_per_batch),
    sampler=SamplerWithoutReplacement(),
)

In [39]:
# PPO requires some “advantage estimation” to be computed.

advantage_module = GAE(
    gamma=gamma, lmbda=lmbda, value_network=value_module, average_gae=True
)

loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=clip_epsilon,
    entropy_bonus=bool(entropy_eps),
    entropy_coef=entropy_eps,
    critic_coef=1.0,
    loss_critic_type="smooth_l1"
)

optim = torch.optim.Adam(loss_module.parameters(), lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim, total_frames // frames_per_batch, 0.0
)

In [ ]:
logs = defaultdict(list)
pbar = tqdm(total=total_frames)
eval_str = ""

for i, tensordict_data in enumerate(collector):

    for _ in range(num_epochs):
        advantage_module(tensordict_data)
        data_view = tensordict_data.reshape(-1)